# CNN From Scratch: Forward and Backward Passes on MNIST

This notebook walks through a **convolutional neural network implemented entirely with manual forward and backward passes**, no `loss.backward()`, no autograd for the layer gradients. Every gradient is computed by hand.

**Goal:** MNIST digit classification using the architecture Conv → BN → ReLU → Conv → BN → ReLU → Linear (10 classes), where the convolution and batch-norm backward passes are written from scratch.

All core implementations live in [`cnn_from_scratch.py`](cnn_from_scratch.py); this notebook imports, verifies, and trains them.

> **Origin note:** These implementations were originally developed as coursework for CPEN 455 (Deep Learning) at UBC and have been refactored into standalone, importable modules with classes and docstrings.

In [ ]:
import sys, time
sys.path.insert(0, ".")

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms

from cnn_from_scratch import Conv2d, ReLU, BatchNorm2d, CNN

print(f"PyTorch {torch.__version__}  |  Device: cpu")

## Architecture Overview

```
Input (B, 1, 28, 28)            ← grayscale MNIST images
  │
  ├─ Conv2d(1→2, 3×3, pad=1)    ← im2col forward + vectorised im2col backward
  ├─ BatchNorm2d                ← manual per-channel normalisation
  ├─ ReLU                       ← manual element-wise gating
  │   → (B, 2, 28, 28)
  │
  ├─ Conv2d(2→2, 3×3, pad=1)
  ├─ BatchNorm2d
  ├─ ReLU
  │   → (B, 2, 28, 28)
  │
  ├─ Flatten                    → (B, 1568)
  └─ Linear(1568, 10)           ← plain matrix multiply
      → (B, 10) logits
```

**What is from scratch:** Conv2d forward (im2col matrix construction + matmul), Conv2d backward (gradient via im2col + col2im scatter), ReLU forward/backward, BatchNorm2d forward/backward, and the full CNN chain-rule backward pass.

**What uses PyTorch:** Tensor operations (`torch.zeros`, `@`, slicing,`torch.bmm`), `torch.softmax` for the cross-entropy gradient, and `torchvision` for MNIST loading. The training loop uses manual SGD, no `torch.optim`.

### About the backward pass

The backward pass for Conv2d reuses the same im2col representation from the forward pass. The filter gradient is a batch matmul between the upstream gradient and the cached im2col matrix. The input gradient is computed by a reverse matmul followed by a **col2im** scatter that places each gradient patch back into the padded input grid.

This vectorised formulation is mathematically identical to the element-wise nested-loop version (which was verified during development against both PyTorch autograd and finite-difference gradient checking) but runs ~1000× faster by replacing 7-deep Python loops with matrix operations.

## Verification: Our Conv2d Matches PyTorch

Before training, we confirm the implementation is numerically correct by comparing against `F.conv2d` (forward) and `torch.autograd.grad` (backward). We use `float64` for tight tolerances.

In [ ]:
torch.set_default_dtype(torch.float64)
torch.manual_seed(42)

B, C, H, W = 2, 1, 8, 8
K, D, P = 3, 2, 1

x = torch.randn(B, C, H, W)
w = torch.randn(D, C, K, K)

# Our implementation
conv = Conv2d(in_channels=C, out_channels=D, kernel_size=K, stride=1, padding=P)
conv.weight = w
y_ours = conv.forward(x)

# PyTorch reference
y_ref = F.conv2d(x, w, stride=1, padding=P)

diff = (y_ours - y_ref).norm().item()
print(f"Forward difference: {diff:.2e}")
assert diff < 1e-10, f"FORWARD MISMATCH: {diff}"
print("✓ Conv2d forward matches F.conv2d")

In [ ]:
x = torch.randn(B, C, H, W, requires_grad=True)
w = torch.randn(D, C, K, K, requires_grad=True)

# PyTorch autograd reference
y_ref = F.conv2d(x, w, stride=1, padding=P)
v = torch.randn_like(y_ref)
grad_x_ref = torch.autograd.grad(y_ref, x, grad_outputs=v, retain_graph=True)[0]
grad_w_ref = torch.autograd.grad(y_ref, w, grad_outputs=v)[0]

# Our backward
conv = Conv2d(in_channels=C, out_channels=D, kernel_size=K, stride=1, padding=P)
conv.weight = w.detach()
_ = conv.forward(x.detach())
grad_x_ours, grad_w_ours = conv.backward(v)

dx = (grad_x_ours - grad_x_ref).norm().item()
dw = (grad_w_ours - grad_w_ref).norm().item()
print(f"grad_input  difference: {dx:.2e}")
print(f"grad_weight difference: {dw:.2e}")
assert dx < 1e-5 and dw < 1e-5
print("✓ Conv2d backward matches autograd")

# Reset to float32 for training speed
torch.set_default_dtype(torch.float32)

## Dataset: MNIST

We load a subset of MNIST (5 000 train / 1 000 test) to keep runtime practical while still allowing the model to learn meaningful features. With only D=2 filters, accuracy will be modest — the point here is demonstrating that the manual gradients produce a model that **actually learns**, not chasing state-of-the-art.

MNIST is auto-downloaded via torchvision on first run; requires internet access (or a pre-existing `./data/` directory).

In [ ]:
TRAIN_SIZE = 5000
TEST_SIZE  = 1000
BATCH_SIZE = 64

transform = transforms.ToTensor()   # scales [0,255] → [0,1], float32

train_full = datasets.MNIST("./data", train=True,  download=True, transform=transform)
test_full  = datasets.MNIST("./data", train=False, download=True, transform=transform)

# Take subsets
train_set = torch.utils.data.Subset(train_full, range(TRAIN_SIZE))
test_set  = torch.utils.data.Subset(test_full,  range(TEST_SIZE))

train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_set)} images  |  Test: {len(test_set)} images")
print(f"Batch size: {BATCH_SIZE}  |  Batches/epoch: {len(train_loader)}")

# Show a few samples
fig, axes = plt.subplots(1, 8, figsize=(12, 1.5))
for i, ax in enumerate(axes):
    img, label = train_full[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(str(label), fontsize=10)
    ax.axis("off")
fig.suptitle("MNIST samples", fontsize=12)
plt.tight_layout()
plt.show()

## Training

**Manual SGD** with a fixed learning rate. The training loop:

1. **Forward:** `CNN.forward()` runs both conv layers, batch norms, ReLUs, and the linear readout, all using our from-scratch code.
2. **Loss:** standard cross-entropy.  We compute its **gradient w.r.t. logits analytically**: for averaged CE loss, ∂L/∂z = (softmax(z) − one_hot(y)) / B. This is a standard result, no autograd needed.
3. **Backward:** `CNN.backward()` propagates the gradient through every layer using our manual backward methods.
4. **Update:** plain SGD weight subtraction.

In [ ]:
# -- Hyperparameters --
NUM_EPOCHS = 5
LR = 0.01
D = 2          # number of conv filters per layer
K = 3          # kernel size
P = 1          # padding (preserves spatial dims with K=3, stride=1)

# -- Initialise weights --
torch.manual_seed(0)
filter_1 = torch.randn(D, 1, K, K) * 0.1       # Conv1: 1 input channel
filter_2 = torch.randn(D, D, K, K) * 0.1       # Conv2: D input channels
weight   = torch.randn(D * 28 * 28, 10) * 0.01 # Linear: 1568 → 10

cnn = CNN(in_channels=1, num_filters=D, kernel_size=K, stride=1, padding=P)

# -- Cross-entropy gradient (analytical) --
def cross_entropy_gradient(logits, labels):
    """∂L/∂z = (softmax(z) - one_hot(y)) / B  for mean-reduced CE."""
    B = logits.shape[0]
    probs = torch.softmax(logits, dim=1)
    one_hot = torch.zeros_like(probs)
    one_hot[torch.arange(B), labels] = 1.0
    return (probs - one_hot) / B

# -- Tracking --
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

# -- Evaluation helper --
def evaluate(loader, f1, f2, w):
    total_loss, correct, total = 0.0, 0, 0
    eval_cnn = CNN(in_channels=1, num_filters=D, kernel_size=K, stride=1, padding=P)
    with torch.no_grad():
        for imgs, labels in loader:
            logits = eval_cnn.forward(imgs, f1, f2, w)
            total_loss += F.cross_entropy(logits, labels, reduction="sum").item()
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.shape[0]
    return total_loss / total, correct / total

# -- Training loop --
print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>10}  {'Test Acc':>10}  {'Time':>8}")
print("─" * 52)

total_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0
    t0 = time.time()

    for imgs, labels in train_loader:
        B = imgs.shape[0]

        # 1) Forward
        logits = cnn.forward(imgs, filter_1, filter_2, weight)

        # 2) Loss (for tracking only)
        loss_val = F.cross_entropy(logits, labels).item()
        epoch_loss += loss_val * B
        epoch_correct += (logits.argmax(dim=1) == labels).sum().item()
        epoch_total += B

        # 3) Backward, manual gradient through the full network
        grad_logits = cross_entropy_gradient(logits.detach(), labels)
        gf1, gf2, gw = cnn.backward(grad_logits)

        # 4) SGD update
        filter_1 = filter_1 - LR * gf1
        filter_2 = filter_2 - LR * gf2
        weight   = weight   - LR * gw

    # -- Epoch stats --
    train_loss = epoch_loss / epoch_total
    train_acc  = epoch_correct / epoch_total
    _, test_acc = evaluate(test_loader, filter_1, filter_2, weight)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_acc"].append(test_acc)

    elapsed = time.time() - t0
    print(f"{epoch:>5d}  {train_loss:>10.4f}  {train_acc:>10.1%}  {test_acc:>10.1%}  {elapsed:>7.1f}s")

total_time = time.time() - total_start
print(f"\nTotal training time: {total_time:.1f}s")

## Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

epochs = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs, history["train_loss"], "o-", label="Train loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-entropy loss")
ax1.set_title("Training Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history["train_acc"],  "o-", label="Train accuracy")
ax2.plot(epochs, history["test_acc"],   "s--", label="Test accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print(f"Final train accuracy: {history['train_acc'][-1]:.1%}")
print(f"Final test accuracy:  {history['test_acc'][-1]:.1%}")

In [ ]:
# Show predictions on a batch of test images
eval_cnn = CNN(in_channels=1, num_filters=D, kernel_size=K, stride=1, padding=P)
test_imgs, test_labels = next(iter(test_loader))

with torch.no_grad():
    preds = eval_cnn.forward(test_imgs, filter_1, filter_2, weight).argmax(dim=1)

n_show = min(16, len(test_imgs))
fig, axes = plt.subplots(2, 8, figsize=(14, 3.5))

for i, ax in enumerate(axes.flat[:n_show]):
    ax.imshow(test_imgs[i].squeeze(), cmap="gray")
    color = "green" if preds[i] == test_labels[i] else "red"
    ax.set_title(f"pred={preds[i].item()}\ntrue={test_labels[i].item()}",
                 fontsize=9, color=color)
    ax.axis("off")

for ax in axes.flat[n_show:]:
    ax.axis("off")

fig.suptitle("Test Predictions  (green = correct, red = wrong)", fontsize=12)
plt.tight_layout()
plt.show()

correct_shown = (preds[:n_show] == test_labels[:n_show]).sum().item()
print(f"Shown: {correct_shown}/{n_show} correct")

## Some things I Learned

1. **Backward passes are where the understanding lives.** Writing `loss.backward()` hides an enormous amount of work. Implementing the conv gradient by hand, and then realising the im2col matrix connects forward and backward, made the chain rule feel concrete.

2. **Vectorisation is non-negotiable.** The initial element-wise implementation (7-deep nested loops in Python) took ~56 seconds per batch on 28×28 images. Rewriting the backward to reuse the im2col matrix with batch matmuls brought it down to ~0.04 seconds: a ~1 400× speedup, while producing numerically identical gradients. The mathematical insight: ∂L/∂filter is just `grad_out @ im2col`, and ∂L/∂input is `col2im(grad_out^T @ filter_rows)`.

3. **im2col is elegant but memory-hungry.** Rearranging input patches into a matrix converts convolution into a single matmul, which is why it's fast on GPUs. The tradeoff: the im2col matrix can be much larger than the original input (each pixel appears in up to K² patches).

4. **BatchNorm's backward pass is surprisingly subtle.** The mean and variance depend on *all* elements in the (B, H, W) slice, so the chain rule branches into three terms. Getting this wrong is a common source (at least for me) of silent training failures.

5. **Manual gradient verification matters.** Comparing analytical gradients against both PyTorch autograd and finite-difference approximations (implemented in `gradient_check()`) gave high confidence before committing to a training run.

### Limitations

- **D=2 filters** is far too few for competitive accuracy; a practical CNN would use 32–128 filters per layer.
- **Training subset** (5,000 images) keeps runtime under ~1 minute.
- **No pooling, dropout, or data augmentation**, minimal architecture for clarity.
- **Only stride=1 with square kernels** is supported; a production implementation would generalise to arbitrary strides, dilation, and rectangular kernels and all other tricks seen in the lectures.